In [39]:
import os
import sys
import json
import time
import kaggle
from kagglehub.competition import competition_download

import numpy as np
import pandas as pd

os.environ['KAGGLE_USERNAME'] = json.load(open('/home/osman/.config/kaggle/kaggle.json'))['username']
os.environ['KAGGLE_KEY'] = json.load(open('/home/osman/.config/kaggle/kaggle.json'))['key']
path = competition_download('ing-hubs-turkiye-datathon')

In [40]:
customer_history = pd.read_csv(f"{path}/customer_history.csv")
customers = pd.read_csv(f"{path}/customers.csv")
referance_data = pd.read_csv(f"{path}/referance_data.csv")
referance_data_test = pd.read_csv(f"{path}/referance_data_test.csv")
sample_submission = pd.read_csv(f"{path}/sample_submission.csv") 

In [41]:
referance_data_test["cust_id"].value_counts().sort_values(ascending=False)

cust_id
1         1
2         1
9         1
15        1
19        1
         ..
199951    1
199952    1
199963    1
199964    1
199979    1
Name: count, Length: 43006, dtype: int64

In [42]:
referance_data["cust_id"].value_counts().sort_values(ascending=False)

cust_id
0         1
3         1
5         1
6         1
7         1
         ..
199995    1
199996    1
199997    1
199998    1
199999    1
Name: count, Length: 133287, dtype: int64

farklı zaman damgalararında aynı müşteriye ait hiç gözlem yok hem train hem test setinde bu zaman bağımsız çalışmamızı rahatlatır

In [43]:
customers.head()

,cust_id,gender,age,province,religion,work_type,work_sector,tenure
0,0,F,64,NOH,U,Part-time,Technology,135
1,1,F,57,ZUI,O,Full-time,Finance,65
2,2,F,62,NOB,M,Self-employed,Healthcare,224
3,3,F,22,ZUI,C,Student,NaN,47
4,5,M,27,ZUI,U,Full-time,Finance,108


In [44]:
set(referance_data_test["cust_id"]) - set(customer_history["cust_id"]) 

set()

In [45]:
len(set(customer_history["cust_id"]) - set(referance_data_test["cust_id"]) )

133287

test setinde olup history de olmayan yok yani yeni müşterinin gelmediğini varsayıyoruz
history de olan ama test de olmaya 133287 gözlem var

In [46]:
customers.isna().sum()

cust_id            0
gender             0
age                0
province           0
religion           0
work_type          0
work_sector    30134
tenure             0
dtype: int64

work_sector de nan değerler var neden kaynaklandığını inceleyelim

In [47]:
customers[customers["work_sector"].isna()]["work_type"].value_counts()

work_type
Retired       13306
Unemployed    12310
Student        4518
Name: count, dtype: int64

açıkca belli ki sektörü olmayan kişiler aslında çalışmadıkları için sektörü yok yanlış girdi sözkonusu değil bu durumda doğrudan work_type ile dolduracağım

In [49]:
customers["work_sector"] = customers["work_sector"].fillna(customers["work_type"])

In [50]:
customers

,cust_id,gender,age,province,religion,work_type,work_sector,tenure
0,0,F,64,NOH,U,Part-time,Technology,135
1,1,F,57,ZUI,O,Full-time,Finance,65
2,2,F,62,NOB,M,Self-employed,Healthcare,224
3,3,F,22,ZUI,C,Student,Student,47
4,5,M,27,ZUI,U,Full-time,Finance,108
...,...,...,...,...,...,...,...,...
176288,199995,F,54,GEL,C,Part-time,Public Sector,217
176289,199996,M,47,GEL,C,Full-time,Public Sector,37
176290,199997,F,66,NOB,C,Retired,Retired,227
176291,199998,F,31,ZUI,U,Self-employed,Education,156


In [58]:
customers["cust_age_month"] = (customers["age"] * 12)  - customers["tenure"]

In [59]:
customers[customers["cust_age_month"] / 12 < 18]

,cust_id,gender,age,province,religion,work_type,work_sector,tenure,cust_age_month
51,57,M,20,ZUI,M,Full-time,Finance,34,206
55,61,M,20,NOB,O,Student,Student,35,205
79,90,F,20,ZUI,U,Student,Student,41,199
89,102,F,20,FRI,C,Unemployed,Unemployed,30,210
114,128,F,21,OVE,C,Part-time,Education,38,214
...,...,...,...,...,...,...,...,...,...
176024,199696,F,20,UTR,J,Student,Student,35,205
176127,199816,F,20,UTR,U,Part-time,Education,35,205
176193,199888,F,21,NOH,O,Student,Student,37,215
176245,199947,F,21,GRO,J,Part-time,Retail,37,215


bankacılıkta müşteri olabilmek için 18 üstü olman gerek bu durumda 18 altında üye olmuş gibi gözüken satırları kaldırmalıyız ama belkide başka ülkelerde bu durum normaldir

In [61]:
customers[customers["work_type"] == "Retired"]["age"].describe()

count    13306.000000
mean        69.786938
std          4.431965
min         65.000000
25%         66.000000
50%         69.000000
75%         72.000000
max        100.000000
Name: age, dtype: float64

In [62]:
customers["age"].describe()

count    176293.000000
mean         43.638091
std          14.551524
min          19.000000
25%          31.000000
50%          43.000000
75%          55.000000
max         100.000000
Name: age, dtype: float64

In [67]:
customers.groupby("religion").agg({"age":"median"})

,age
religion,
C,43.0
J,43.0
M,43.0
O,43.0
U,43.0


In [56]:
customers["age"].min()

np.int64(19)

In [69]:
customer_history.isna().sum()

cust_id                             0
date                                0
mobile_eft_all_cnt             112334
active_product_category_nbr         0
mobile_eft_all_amt             112334
cc_transaction_all_amt         166746
cc_transaction_all_cnt         166746
dtype: int64

In [80]:
customer_history[customer_history["mobile_eft_all_cnt"].isna()].drop("date", axis=1).describe()

,cust_id,mobile_eft_all_cnt,active_product_category_nbr,mobile_eft_all_amt,cc_transaction_all_amt,cc_transaction_all_cnt
count,112334.000000,0.0,112334.000000,0.0,112334.000000,112334.000000
mean,98756.897066,NaN,2.689061,NaN,556.339753,20.260901
std,57351.937431,NaN,0.536825,NaN,1071.606928,23.178861
min,77.000000,NaN,2.000000,NaN,0.000000,0.000000
25%,48877.000000,NaN,2.000000,NaN,16.490000,3.000000
50%,99231.000000,NaN,3.000000,NaN,102.670000,13.000000
75%,148111.000000,NaN,3.000000,NaN,598.462500,30.000000
max,199944.000000,NaN,5.000000,NaN,16478.280000,267.000000


In [87]:
customer_history[customer_history["cust_id"] == 0]

,cust_id,date,mobile_eft_all_cnt,active_product_category_nbr,mobile_eft_all_amt,cc_transaction_all_amt,cc_transaction_all_cnt
0,0,2016-01-01,1.0,2,151.20,NaN,NaN
1,0,2016-02-01,1.0,2,178.70,NaN,NaN
2,0,2016-03-01,2.0,2,37.38,NaN,NaN
3,0,2016-04-01,4.0,2,100.90,NaN,NaN
4,0,2016-05-01,3.0,3,132.28,NaN,NaN
5,0,2016-06-01,1.0,2,79.86,NaN,NaN
6,0,2016-07-01,1.0,2,121.27,NaN,NaN
7,0,2016-08-01,4.0,2,31.54,NaN,NaN
8,0,2016-09-01,4.0,2,93.80,NaN,NaN
9,0,2016-10-01,5.0,2,52.01,NaN,NaN


In [83]:
referance_data[referance_data["cust_id"] == 77]

,cust_id,ref_date,churn
52,77,2018-03-01,0


In [78]:
customers[customers["cust_id"] == 77]

,cust_id,gender,age,province,religion,work_type,work_sector,tenure,cust_age_month
70,77,F,33,FLE,M,Full-time,Education,180,216


In [71]:
customer_history[customer_history["mobile_eft_all_cnt"]==0]

,cust_id,date,mobile_eft_all_cnt,active_product_category_nbr,mobile_eft_all_amt,cc_transaction_all_amt,cc_transaction_all_cnt
57,1,2019-01-01,0.0,3,0.0,72.72,11.0
58,1,2019-02-01,0.0,3,0.0,15.74,8.0
195,7,2016-11-01,0.0,3,0.0,3210.39,59.0
196,7,2016-12-01,0.0,3,0.0,3229.64,73.0
197,7,2017-01-01,0.0,3,0.0,5374.94,96.0
...,...,...,...,...,...,...,...
5359604,199999,2018-02-01,0.0,2,0.0,9.66,1.0
5359605,199999,2018-03-01,0.0,2,0.0,22.89,2.0
5359606,199999,2018-04-01,0.0,2,0.0,48.96,3.0
5359607,199999,2018-05-01,0.0,2,0.0,38.20,2.0


In [92]:
history_statics = customer_history.drop("date", axis=1).groupby("cust_id").agg(["min", "max", "median", "mean"])
history_statics.columns = ["_".join(col).strip() for col in history_statics.columns.values]


In [93]:
history_statics

,mobile_eft_all_cnt_min,mobile_eft_all_cnt_max,mobile_eft_all_cnt_median,mobile_eft_all_cnt_mean,active_product_category_nbr_min,active_product_category_nbr_max,active_product_category_nbr_median,active_product_category_nbr_mean,mobile_eft_all_amt_min,mobile_eft_all_amt_max,mobile_eft_all_amt_median,mobile_eft_all_amt_mean,cc_transaction_all_amt_min,cc_transaction_all_amt_max,cc_transaction_all_amt_median,cc_transaction_all_amt_mean,cc_transaction_all_cnt_min,cc_transaction_all_cnt_max,cc_transaction_all_cnt_median,cc_transaction_all_cnt_mean
cust_id,,,,,,,,,,,,,,,,,,,,
0,1.0,5.0,2.0,2.238095,2,3,2.0,2.047619,31.54,297.70,121.270,122.782857,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0.0,5.0,1.0,1.605263,2,3,3.0,2.973684,0.00,83.03,12.360,17.424737,10.55,979.15,196.620,229.945263,3.0,35.0,13.0,14.289474
2,1.0,5.0,2.0,2.351351,2,3,3.0,2.837838,13.41,1436.46,556.730,609.999189,0.00,19.52,11.840,9.323243,0.0,7.0,2.0,1.972973
3,1.0,4.0,1.0,1.676471,3,3,3.0,3.000000,1.67,498.99,25.340,72.665000,121.68,1749.13,428.285,629.045882,11.0,49.0,21.0,24.558824
5,1.0,6.0,2.0,2.555556,2,2,2.0,2.000000,64.56,1390.78,220.270,386.442593,10.69,829.67,15.710,91.887778,1.0,48.0,4.0,13.074074
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
199995,0.0,6.0,1.0,1.393939,3,3,3.0,3.000000,0.00,1094.64,9.160,113.774848,10.30,1107.12,126.010,221.495455,1.0,95.0,32.0,32.000000
199996,1.0,5.0,2.0,2.000000,2,3,3.0,2.800000,24.17,791.36,91.895,159.268000,0.00,19.29,15.275,12.627333,0.0,4.0,2.0,1.966667
199997,1.0,22.0,7.5,9.055556,2,2,2.0,2.000000,1386.33,7651.39,3592.045,3753.172222,5.38,1142.98,134.670,230.284722,3.0,32.0,12.0,14.944444


In [ ]:
customers_with_history = customers.merge(history_statics, "left", on="cust_id")

,cust_id,gender,age,province,religion,work_type,work_sector,tenure,cust_age_month,mobile_eft_all_cnt_min,...,mobile_eft_all_amt_median,mobile_eft_all_amt_mean,cc_transaction_all_amt_min,cc_transaction_all_amt_max,cc_transaction_all_amt_median,cc_transaction_all_amt_mean,cc_transaction_all_cnt_min,cc_transaction_all_cnt_max,cc_transaction_all_cnt_median,cc_transaction_all_cnt_mean
0,0,F,64,NOH,U,Part-time,Technology,135,633,1.0,...,121.270,122.782857,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,F,57,ZUI,O,Full-time,Finance,65,619,0.0,...,12.360,17.424737,10.55,979.15,196.620,229.945263,3.0,35.0,13.0,14.289474
2,2,F,62,NOB,M,Self-employed,Healthcare,224,520,1.0,...,556.730,609.999189,0.00,19.52,11.840,9.323243,0.0,7.0,2.0,1.972973
3,3,F,22,ZUI,C,Student,Student,47,217,1.0,...,25.340,72.665000,121.68,1749.13,428.285,629.045882,11.0,49.0,21.0,24.558824
4,5,M,27,ZUI,U,Full-time,Finance,108,216,1.0,...,220.270,386.442593,10.69,829.67,15.710,91.887778,1.0,48.0,4.0,13.074074
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
176288,199995,F,54,GEL,C,Part-time,Public Sector,217,431,0.0,...,9.160,113.774848,10.30,1107.12,126.010,221.495455,1.0,95.0,32.0,32.000000
176289,199996,M,47,GEL,C,Full-time,Public Sector,37,527,1.0,...,91.895,159.268000,0.00,19.29,15.275,12.627333,0.0,4.0,2.0,1.966667
176290,199997,F,66,NOB,C,Retired,Retired,227,565,1.0,...,3592.045,3753.172222,5.38,1142.98,134.670,230.284722,3.0,32.0,12.0,14.944444
176291,199998,F,31,ZUI,U,Self-employed,Education,156,216,1.0,...,575.580,634.658077,10.20,534.49,49.245,114.740769,1.0,50.0,25.5,19.923077
